# 🎙️ Dorina HiFi-GAN V1: 300k Adım Kaggle Üretim Eğitimi
Bu notebook, **Claude'un tüm üretim kurallarıyla** hazırlanmıştır:
- 2.390 adet Stüdyo Eğitim Klibi
- 50 adet Ayrı Stüdyo Validation Klibi (Eğitime asla girmez)
- Atomik Checkpoint Yazma (Yarım kalma bozulmalarına karşı korumalı)
- 11. Saat Otomatik Güvenli Çıkış
- Her 5.000 adımda sabit 5 validation klibi üzerinden WAV üretimi

**GPU Seçimi:** Sağ üst menüden `Settings -> Accelerator -> GPU T4 x2` veya `P100` seçin.

In [ ]:
# 1. Kurulum
!pip install -q soundfile torchaudio pandas matplotlib

In [ ]:
# 2. Tek Standart Mel-Spektrogram Fonksiyonu (22.050 Hz Saf Standart)
import os, random, time, io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import pandas as pd
import numpy as np
from IPython.display import Audio, display

class MelConfig:
    sample_rate: int = 22050
    n_fft: int = 1024
    win_length: int = 1024
    hop_length: int = 256
    n_mels: int = 80
    f_min: float = 0.0
    f_max: float = 8000.0

class MelSpectrogramExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_fft=1024, win_length=1024, hop_length=256,
            f_min=0.0, f_max=8000.0, n_mels=80, power=1.0, normalized=False, center=False
        )
    def forward(self, wav):
        if wav.dim() == 1: wav = wav.unsqueeze(0)
        mel = self.mel_transform(wav)
        return torch.log(torch.clamp(mel, min=1e-5))

mel_fn = MelSpectrogramExtractor().cuda()
print("Mel modülü hazır!")

In [ ]:
# 3. HiFi-GAN Mimarisi (MRF Generator + MPD + MSD)
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilations=[1, 3, 5]):
        super().__init__()
        self.c1 = nn.ModuleList([nn.Conv1d(channels, channels, kernel_size, d, (kernel_size*d-d)//2) for d in dilations])
        self.c2 = nn.ModuleList([nn.Conv1d(channels, channels, kernel_size, 1, (kernel_size-1)//2) for _ in dilations])
    def forward(self, x):
        for c1, c2 in zip(self.c1, self.c2):
            res = x
            x = F.leaky_relu(x, 0.1)
            x = c1(x)
            x = F.leaky_relu(x, 0.1)
            x = c2(x) + res
        return x

class HiFiGANGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_pre = nn.Conv1d(80, 512, 7, 1, 3)
        self.ups = nn.ModuleList([
            nn.ConvTranspose1d(512, 256, 16, 8, 4),
            nn.ConvTranspose1d(256, 128, 16, 8, 4),
            nn.ConvTranspose1d(128, 64, 4, 2, 1),
            nn.ConvTranspose1d(64, 32, 4, 2, 1),
        ])
        self.res = nn.ModuleList([
            ResBlock(256, 3), ResBlock(256, 7), ResBlock(256, 11),
            ResBlock(128, 3), ResBlock(128, 7), ResBlock(128, 11),
            ResBlock(64, 3), ResBlock(64, 7), ResBlock(64, 11),
            ResBlock(32, 3), ResBlock(32, 7), ResBlock(32, 11),
        ])
        self.conv_post = nn.Conv1d(32, 1, 7, 1, 3)

    def forward(self, x):
        x = self.conv_pre(x)
        r_idx = 0
        for up in self.ups:
            x = F.leaky_relu(x, 0.1)
            x = up(x)
            xs = (self.res[r_idx](x) + self.res[r_idx+1](x) + self.res[r_idx+2](x)) / 3.0
            r_idx += 3
            x = xs
        x = F.leaky_relu(x, 0.1)
        return torch.tanh(self.conv_post(x)).squeeze(1)

class DiscriminatorP(nn.Module):
    def __init__(self, period):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            nn.Conv2d(1, 32, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(32, 128, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(128, 512, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(512, 1024, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(1024, 1024, (5, 1), 1, padding=(2, 0)),
        ])
        self.post = nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0))
    def forward(self, x):
        b, t = x.shape
        if t % self.period != 0:
            x = F.pad(x, (0, self.period - (t % self.period)), 'reflect')
        x = x.view(b, 1, -1, self.period)
        fmap = []
        for c in self.convs:
            x = F.leaky_relu(c(x), 0.1); fmap.append(x)
        return torch.flatten(self.post(x), 1, -1), fmap

class MPD(nn.Module):
    def __init__(self):
        super().__init__()
        self.discs = nn.ModuleList([DiscriminatorP(p) for p in [2, 3, 5, 7, 11]])
    def forward(self, y, y_g):
        r_outs, g_outs, f_r, f_g = [], [], [], []
        for d in self.discs:
            ro, fr = d(y); go, fg = d(y_g)
            r_outs.append(ro); g_outs.append(go); f_r.append(fr); f_g.append(fg)
        return r_outs, g_outs, f_r, f_g

print("HiFi-GAN Mimarisi hazır!")

In [ ]:
# 4. Veri Yükleme (Otomatik Hugging Face Parquet Çekici)
# 2.390 adet gerçek stüdyo kadın sesini hazırlar
os.makedirs("dataset/train", exist_ok=True)
os.makedirs("dataset/val", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("val_outputs", exist_ok=True)

repos = [f"falan42/Turkish-Bentropi-tts-dataset-part-{i}" for i in range(2, 12)]
total = 0
for r in repos:
    url = f"https://huggingface.co/datasets/{r}/resolve/main/data/train-00000-of-00001.parquet"
    try:
        df = pd.read_parquet(url)
        for _, row in df.iterrows():
            ab = row['audio']['bytes']
            w, sr = sf.read(io.BytesIO(ab))
            t_w = torch.tensor(w, dtype=torch.float32)
            if t_w.dim() > 1: t_w = t_w.t()[:1]
            else: t_w = t_w.unsqueeze(0)
            if sr != 22050:
                t_w = torchaudio.transforms.Resample(sr, 22050)(t_w)
            data_np = t_w.squeeze(0).numpy()
            peak = np.max(np.abs(data_np)) + 1e-8
            data_np = (data_np / peak) * 0.95
            
            # İlk 50 klip -> Validation (Eğitimden tamamen izole)
            if total < 50:
                sf.write(f"dataset/val/val_{total:04d}.wav", data_np, 22050)
            else:
                sf.write(f"dataset/train/train_{total-50:05d}.wav", data_np, 22050)
            total += 1
        print(f"[+] {r} indirildi. Toplam ses: {total}")
    except Exception:
        pass

train_files = [os.path.join("dataset/train", f) for f in os.listdir("dataset/train") if f.endswith(".wav")]
val_files = [os.path.join("dataset/val", f) for f in os.listdir("dataset/val") if f.endswith(".wav")][:5] # Sabit 5 test sesi
print(f"Veri Hazır: {len(train_files)} Eğitim | {len(val_files)} Sabit Validation Klibi")

In [ ]:
# 5. Üretim Eğitim Döngüsü (300k Adım, Atomik Checkpoint, 11. Saat Koruması)
generator = HiFiGANGenerator().cuda()
mpd = MPD().cuda()

opt_g = torch.optim.AdamW(generator.parameters(), lr=2e-4, betas=(0.8, 0.99), weight_decay=1e-4)
opt_d = torch.optim.AdamW(mpd.parameters(), lr=2e-4, betas=(0.8, 0.99), weight_decay=1e-4)

start_step = 1
TOTAL_STEPS = 300000
SEGMENT_SIZE = 8192
BATCH_SIZE = 16

# Varsa checkpoint yükle (Oturumlar arası kesintisiz devam)
CKPT_FILE = "checkpoints/dorina_hifigan_active.pt"
if os.path.exists(CKPT_FILE):
    ckpt = torch.load(CKPT_FILE)
    generator.load_state_dict(ckpt['generator'])
    mpd.load_state_dict(ckpt['mpd'])
    opt_g.load_state_dict(ckpt['opt_g'])
    opt_d.load_state_dict(ckpt['opt_d'])
    start_step = ckpt['step'] + 1
    print(f"[*] Önceki oturumdan devam ediliyor: Adım {start_step}")

start_time = time.time()
MAX_HOURS = 11.0 # 11. saatte güvenli kaydet ve çık

generator.train(); mpd.train()
print(f"[*] Eğitim Adım {start_step}'den başladı...")

for step in range(start_step, TOTAL_STEPS + 1):
    # 11. Saat Koruması
    elapsed_hours = (time.time() - start_time) / 3600.0
    if elapsed_hours >= MAX_HOURS:
        print(f"[!] 11 saatlik oturum süresi doldu. Güvenli checkpoint kaydedilip çıkılıyor...")
        break
        
    # Batch oluştur
    batch = []
    for _ in range(BATCH_SIZE):
        fn = random.choice(train_files)
        d, _ = sf.read(fn)
        if len(d) > SEGMENT_SIZE + 500:
            st = random.randint(0, len(d) - SEGMENT_SIZE)
            batch.append(d[st:st+SEGMENT_SIZE])
        else:
            batch.append(np.pad(d, (0, max(0, SEGMENT_SIZE - len(d))))[:SEGMENT_SIZE])
    
    wav_batch = torch.tensor(np.array(batch), dtype=torch.float32).cuda()
    with torch.no_grad():
        mel_gt = mel_fn(wav_batch)
        
    # Generator üretim
    wav_gen = generator(mel_gt)
    min_l = min(wav_batch.shape[-1], wav_gen.shape[-1])
    wav_real = wav_batch[:, :min_l]
    wav_fake = wav_gen[:, :min_l]
    
    # DISCRIMINATOR
    opt_d.zero_grad()
    r_outs, g_outs, _, _ = mpd(wav_real, wav_fake.detach())
    loss_d = sum(torch.mean((1.0 - ro)**2) + torch.mean(go**2) for ro, go in zip(r_outs, g_outs))
    loss_d.backward()
    opt_d.step()
    
    # GENERATOR (Claude Loss Formülü: Adv + 2*FM + 45*Mel)
    opt_g.zero_grad()
    _, g_outs, f_r, f_g = mpd(wav_real, wav_fake)
    loss_adv = sum(torch.mean((1.0 - dg)**2) for dg in g_outs)
    loss_fm = sum(torch.mean(torch.abs(r-g)) for fr, fg in zip(f_r, f_g) for r, g in zip(fr, fg)) * 2.0
    loss_mel = F.l1_loss(mel_fn(wav_fake), mel_fn(wav_real)) * 45.0
    
    loss_g = loss_adv + loss_fm + loss_mel
    loss_g.backward()
    opt_g.step()
    
    if step % 200 == 0:
        print(f"Step [{step}/{TOTAL_STEPS}] - Loss G: {loss_g.item():.3f} (Mel: {loss_mel.item():.4f}) | Loss D: {loss_d.item():.3f}")
        
    # Her 5.000 adımda bir ATOMİK CHECKPOINT ve VALIDATION SESLERİ
    if step % 5000 == 0:
        generator.eval()
        # Sabit 5 validation klibi dinleme çıktısı
        with torch.no_grad():
            for v_idx, v_path in enumerate(val_files):
                v_data, _ = sf.read(v_path)
                v_mel = mel_fn(torch.tensor(v_data, dtype=torch.float32).cuda())
                v_rec = generator(v_mel).squeeze().cpu().numpy()
                sf.write(f"val_outputs/step_{step}_val_{v_idx}.wav", v_rec, 22050)
        
        # Atomik Checkpoint: Önce geçiciye yaz, sonra atomik taşı (Bozulma sıfır)
        temp_ckpt = "checkpoints/dorina_ckpt_temp.pt"
        torch.save({
            'step': step,
            'generator': generator.state_dict(),
            'mpd': mpd.state_dict(),
            'opt_g': opt_g.state_dict(),
            'opt_d': opt_d.state_dict()
        }, temp_ckpt)
        os.replace(temp_ckpt, CKPT_FILE)
        print(f"[CHECKPOINT] Adım {step} başarıyla atomik olarak mühürrlendi.")
        generator.train()

print("[BİTTİ] Oturum tamamlandı!")

In [ ]:
# 6. Son Validation Sesini Dinle!
last_val = sorted([f for f in os.listdir("val_outputs") if f.endswith(".wav")])
if last_val:
    p = os.path.join("val_outputs", last_val[-1])
    print(f"En son üretilen ses: {p}")
    display(Audio(p, rate=22050))
